In [4]:
import os
import sys
from pathlib import Path

# 1. Lock working directory to main project root (2 levels up from src/notebook/)
current_dir = Path.cwd()
if current_dir.name == "notebook" and current_dir.parent.name == "src":
    project_root = current_dir.parent.parent
    os.chdir(project_root)
elif current_dir.name == "src":
    project_root = current_dir.parent
    os.chdir(project_root)
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Kernel working directory locked to:", Path.cwd())

# 2. Automatically create all necessary project directories
folders = [
    "tests"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All project directories verified/created successfully!")

Kernel working directory locked to: /home/s2025ae05884/workspace/heart-disease-predict
All project directories verified/created successfully!


In [5]:
%%writefile tests/test_pipeline.py
import os
import joblib
import pandas as pd
import numpy as np
from src.preprocess import build_preprocessor, ALL_FEATURES, TARGET_COL

def test_cleaned_data_exists():
    """Verify that the cleaned UCI dataset exists and is non-empty."""
    data_path = "data/processed/heart_disease_cleaned.csv"
    assert os.path.exists(data_path), "Cleaned dataset file not found!"
    df = pd.read_csv(data_path)
    assert len(df) > 0, "Dataset is empty!"
    assert TARGET_COL in df.columns, "Target column missing from dataset!"

def test_preprocessor_transformation():
    """Verify that the ColumnTransformer outputs the correct shape without errors."""
    data_path = "data/processed/heart_disease_cleaned.csv"
    df = pd.read_csv(data_path)
    
    X = df[ALL_FEATURES]
    preprocessor = build_preprocessor()
    X_transformed = preprocessor.fit_transform(X)
    
    # Check that output is a 2D numpy array and has more columns than raw (due to one-hot encoding)
    assert isinstance(X_transformed, np.ndarray)
    assert X_transformed.shape[0] == len(df)
    assert X_transformed.shape[1] > len(ALL_FEATURES)

def test_model_artifact_and_inference():
    """Verify that the saved model artifact loads and returns valid predictions."""
    model_path = "models/heart_disease_pipeline.pkl"
    assert os.path.exists(model_path), "Champion model pickle artifact not found!"
    
    pipeline = joblib.load(model_path)
    
    # Create a synthetic sample matching the 13 input features
    sample_data = pd.DataFrame([{
        "age": 55.0, "sex": 1.0, "cp": 2.0, "trestbps": 140.0, "chol": 240.0,
        "fbs": 0.0, "restecg": 1.0, "thalach": 150.0, "exang": 0.0,
        "oldpeak": 1.5, "slope": 2.0, "ca": 0.0, "thal": 2.0
    }])
    
    prediction = pipeline.predict(sample_data)
    probability = pipeline.predict_proba(sample_data)
    
    # Assertions
    assert prediction[0] in [0, 1], "Prediction must be binary (0 or 1)"
    assert probability.shape == (1, 2), "Probability output shape must be (1, 2)"
    assert 0.0 <= probability[0][1] <= 1.0, "Probability must be between 0 and 1"

Writing tests/test_pipeline.py


In [7]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.2.2, pluggy-1.7.0 -- /home/s2025ae05884/workspace/heart-disease-predict/venv/bin/python
cachedir: .pytest_cache
rootdir: /home/s2025ae05884/workspace/heart-disease-predict
plugins: platformdirs-4.13.3, anyio-4.15.1
collected 3 items                                                              

tests/test_pipeline.py::test_cleaned_data_exists PASSED                  [ 33%]
tests/test_pipeline.py::test_preprocessor_transformation PASSED          [ 66%]
tests/test_pipeline.py::test_model_artifact_and_inference PASSED         [100%]

============================== 3 passed in 6.28s ===============================
